# ਲੈਸਨ 08 - ਬਹੁ-ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ


## ਸੈਟਅਪ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ਕਿਉਂ ਮਲਟੀ-ਏਜੰਟ ਸਿਸਟਮਸ?

ਅਸਲ ਦੀ ਦੁਨੀਆਂ ਦੇ ਕੰਮ ਜਿਵੇਂ ਕਿ ਯਾਤਰਾ ਯੋਜਨਾ ਬਣਾਉਣ ਵਿੱਚ ਕਈ ਕਿਸਮ ਦੇ ਵਿਸ਼ੇਸ਼ਗਿਆਨ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ — ਲੌਜਿਸਟਿਕਸ, ਸਥਾਨਕ ਜਾਣਕਾਰੀ, ਬਜਟਿੰਗ, ਅਤੇ ਹੋਰ। ਇੱਕ ਇੱਕਲੌਤਾ ਏਜੰਟ ਸਭ ਕੁਝ ਸੰਭਾਲਣ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰਦਾ ਹੈ ਤਾਂ ਉਹ ਜਲਦੀ ਹੀ ਅਣਹੋਣ ਵਾਲਾ ਬਣ ਜਾਂਦਾ ਹੈ।

ਮਲਟੀ-ਏਜੰਟ ਸਿਸਟਮ ਇਸਨੂੰ **ਖ਼ਾਸ ਤੌਰ ਤੇ ਮੁਹੱਈਆ ਕਰਵਾਉਂਦੇ ਹਨ**: ਹਰ ਏਜੰਟ ਇੱਕ ਖੇਤਰ ਵਿੱਚ ਧਿਆਨ ਕੇਂਦਰਿਤ ਕਰਦਾ ਹੈ, ਜੋ ਕਿ ਇੱਕ ਆਮ ਤਜਰਬੇਕਾਰ ਨਾਲੋਂ ਵਧੀਆ ਨਤੀਜੇ ਦਿੰਦਾ ਹੈ। ਇਹ **ਸਕੇਲਬਿਲਟੀ** ਨੂੰ ਵੀ ਸੁਧਾਰਦਾ ਹੈ — ਤੁਸੀਂ ਨਵੇਂ ਏਜੰਟ ਜੋੜ ਸਕਦੇ ਹੋ (ਜਿਵੇਂ ਕਿ ਉਡਾਣ ਵਿਸ਼ੇਸ਼ਜ્ઞ, ਰੈਸਟੋਰੈਂਟ ਸਮੀਖਿਆਕਾਰ) ਬਿਨਾਂ ਮੌਜੂਦਾ ਕਾਰਵਾਈ ਨੂੰ ਦੁਬਾਰਾ ਲਿਖੇ। ਏਜੰਟ ਇੱਕ ਰਚਨਾ ਬੱਧ ਪਾਈਪਲਾਈਨ ਰਾਹੀਂ ਇਕੱਠੇ ਕੰਮ ਕਰਦੇ ਹਨ, ਇੱਕ ਤੋਂ ਦੂਜੇ ਨੂੰ ਸੰਦਰਭ ਸਾਂਝਾ ਕਰਦੇ ਹੋਏ।


## ਖਾਸ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ਇੱਕ ਲੜੀਵਾਰ ਵਰਕਫਲੋ ਬਣਾਉਣਾ

`WorkflowBuilder` ਤੁਹਾਨੂੰ ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਨਿਰਦੇਸ਼ਤ ਗ੍ਰਾਫ ਵਿੱਚ ਜੋੜਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ। ਇੱਥੇ ਅਸੀਂ ਇੱਕ ਸਾਦਾ ਦੋ-ਕਦਮਾਂ ਵਾਲੀ ਪਾਈਪਲਾਈਨ ਬਣਾਈ ਹੈ: **TravelPlanner** ਯਾਤਰਾ ਦਾ ਰੂਪਰੇਖਾ ਤਿਆਰ ਕਰਦਾ ਹੈ, ਫਿਰ **TravelConcierge** ਇਸ ਦੀ ਸਮੀਖਿਆ ਕਰਦਾ ਹੈ ਅਤੇ ਇਸ ਨੂੰ ਸੁਧਾਰਦਾ ਹੈ।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ਵਰਕਫਲੋ ਵਿੱਚ ਹੋਰ ਏਜੰਟ ਸ਼ਾਮਲ ਕਰਨਾ

ਮਲਟੀ-ਏਜੰਟ ਪੈਟਰਨ ਦਾ ਸਭ ਤੋਂ ਵੱਡਾ ਫਾਇਦਾ ਇਹ ਹੈ ਕਿ ਇਸ ਨੂੰ ਵਧਾਉਣਾ ਕਿੰਨਾ ਆਸਾਨ ਹੈ। ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ **BudgetReviewer** ਏਜੰਟ ਜੋੜਦੇ ਹਾਂ ਜੋ ਯਾਤਰੀ ਦੇ ਬਜਟ ਦੇ ਮੁਤਾਬਕ ਯੋਜਨਾ ਦੀ ਜਾਂਚ ਕਰਦਾ ਹੈ, ਉਹਨਾਂ ਚੀਜ਼ਾਂ ਨੂੰ ਨਿਸ਼ਾਨ ਲਗਾਉਂਦਾ ਹੈ ਜੋ ਖਰਚਾਂ ਨੂੰ ਸੀਮਾ ਤੋਂ ਵੱਧ ਕਰਨ ਦਾ ਸੰਭਾਵਨਾ ਹੋ ਸਕਦਾ ਹੈ, ਅਤੇ ਪੈਸਾ ਬਚਾਉਣ ਵਾਲੇ ਵਿਕਲਪ ਸੁਝਾਉਂਦਾ ਹੈ। ਵਰਕਫਲੋ ਹੁਣ ਲੜੀਵਾਰ ਤਿੰਨ ਏਜੰਟ ਚਲਾਉਂਦਾ ਹੈ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ:

1. **ਖਾਸ ਏਜੰਟ ਬਣਾਏ ਜਾ ਸਕਦੇ ਹਨ** — ਹਰ ਇੱਕ ਦੀ ਸੂਕਸ਼ਮ ਭੂਮਿਕਾ ਹੁੰਦੀ ਹੈ (ਯੋਜਨਾ ਬਣਾਉਣ, ਸੰਰੱਖਣ, ਬਜਟ ਸਮੀਖਿਆ)।
2. **ਏਜੰਟਾਂ ਨੂੰ ਲੜੀਵਾਰ ਕੰਮਕਾਜ ਵਿੱਚ ਜੋੜੋ** `WorkflowBuilder` ਅਤੇ `add_edge` ਦੀ ਵਰਤੋਂ ਨਾਲ।
3. **ਮਲਟੀ-ਏਜੰਟ ਪਾਈਪਲਾਈਨ ਤੋਂ ਆਉਟਪੁੱਟ ਸਟ੍ਰੀਮ ਕਰੋ**, ਉਸਦੇ ਬਾਰੇ ਟ੍ਰੈਕ ਕਰਦੇ ਹੋਏ ਕਿ ਕਿਹੜਾ ਏਜੰਟ ਬੋਲ ਰਿਹਾ ਹੈ।
4. **ਕਿਸੇ ਕਾਰਜਪੱਧਰ ਨੂੰ ਵਧਾਓ** ਨਵੇਂ ਏਜੰਟ ਜੋੜ ਕੇ ਬਿਨਾਂ ਮੌਜੂਦਾ ਏਜੰਟਾਂ ਨੂੰ ਸੋਧੇ।

ਮਲਟੀ-ਏਜੰਟ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਹਰ ਏਜੰਟ ਨੂੰ ਸਧਾਰਣ ਰੱਖਦਾ ਹੈ ਪਰ ਇਕਕੱਲ੍ਹਾ ਏਜੰਟ ਪਾ ਸਕਣ ਵਾਲੇ ਨਾਲੋਂ ਜ਼ਿਆਦਾ ਸੰਪੂਰਨ ਅਤੇ ਵਿਸਤ੍ਰਿਤ ਨਤੀਜੇ ਦਿੰਦਾ ਹੈ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਇਲਜ਼ਾਮ ਤੋਂ ਮੁਕਤ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਜਾਣੂ ਰਹੋ ਕਿ ਸਵੈਚਲਿਤ ਅਨੁਵਾਦ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸੁੱਝੇ ਪਹਲੂ ਹੋ ਸਕਦੇ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਅਧਿਕਾਰਤ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਗੰਭੀਰ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ انسانੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਨਾਲ ਉੱਠਣ ਵਾਲੇ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀ ਜਾਂ ਮਿਸਅਰਥਨ ਲਈ ਜਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
